# 01 — Data Exploration (Phase 2)

Exploratory analysis of the raw UNSW-NB15 training set.

This notebook loads and validates data using functions from `src/` — it does not reimplement loading or validation logic.

In [ ]:
import sys
from pathlib import Path

# Allow imports from the project root when running this notebook from notebooks/
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_training_data
from src.data.validation import validate_dataset
from src.utils.config import load_config, load_classes
from src.utils.paths import get_results_dir

sns.set_theme(style="whitegrid")
config = load_config()
classes = load_classes()
target_col = config["data"]["target_column"]
binary_col = config["data"]["binary_label_column"]
figures_dir = get_results_dir(config) / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

## 1–2. Load data and re-run validation

Uses `src.data.loader` and `src.data.validation` directly — see `results/reports/data_validation_report.txt` for the full report.

In [ ]:
train_df = load_training_data(config)
train_df.head()

## 3. Dataset shape

In [ ]:
print(f"Rows: {train_df.shape[0]}, Columns: {train_df.shape[1]}")

## 4. Column information

In [ ]:
train_df.info()

## 5. Data type analysis

In [ ]:
train_df.dtypes.value_counts()

## 6. Missing value analysis

In [ ]:
missing = train_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing if not missing.empty else "No missing values."

## 7. Duplicate analysis

In [ ]:
print(f"Duplicate rows: {train_df.duplicated().sum()}")

## 8. Statistical summary

In [ ]:
train_df.describe(include="all").T

## 9. Attack category distribution (`attack_cat`)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order = train_df[target_col].value_counts().index
sns.countplot(data=train_df, y=target_col, order=order, ax=ax)
ax.set_title("Attack Category Distribution")
plt.tight_layout()
fig.savefig(figures_dir / "attack_category_distribution.png", dpi=150)
plt.show()

## 10. Binary label distribution (`label`)

In [ ]:
if binary_col in train_df.columns:
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.countplot(data=train_df, x=binary_col, ax=ax)
    ax.set_title("Binary Label Distribution (0 = normal, 1 = attack)")
    plt.tight_layout()
    fig.savefig(figures_dir / "binary_label_distribution.png", dpi=150)
    plt.show()
else:
    print(f"Column '{binary_col}' not found — skipping.")

## 11. Protocol distribution (`proto`)

In [ ]:
if "proto" in train_df.columns:
    top_proto = train_df["proto"].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(x=top_proto.values, y=top_proto.index, ax=ax)
    ax.set_title("Top 15 Protocols by Count")
    plt.tight_layout()
    fig.savefig(figures_dir / "protocol_distribution.png", dpi=150)
    plt.show()
else:
    print("Column 'proto' not found — skipping.")

## 12. Service distribution (`service`)

In [ ]:
if "service" in train_df.columns:
    top_service = train_df["service"].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(x=top_service.values, y=top_service.index, ax=ax)
    ax.set_title("Top 15 Services by Count")
    plt.tight_layout()
    fig.savefig(figures_dir / "service_distribution.png", dpi=150)
    plt.show()
else:
    print("Column 'service' not found — skipping.")

## 13. Connection state distribution (`state`)

In [ ]:
if "state" in train_df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    order = train_df["state"].value_counts().index
    sns.countplot(data=train_df, y="state", order=order, ax=ax)
    ax.set_title("Connection State Distribution")
    plt.tight_layout()
    fig.savefig(figures_dir / "connection_state_distribution.png", dpi=150)
    plt.show()
else:
    print("Column 'state' not found — skipping.")

## 14. Numerical feature distributions

Plots histograms for up to 12 numerical columns (excluding the target/binary label) to keep the notebook readable — extend `cols_to_plot` for a full sweep.

In [ ]:
numeric_cols = train_df.select_dtypes(include="number").columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in (binary_col,)]
cols_to_plot = numeric_cols[:12]

fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, col in zip(axes.flatten(), cols_to_plot):
    sns.histplot(train_df[col], bins=40, ax=ax, kde=False)
    ax.set_title(col, fontsize=10)
for ax in axes.flatten()[len(cols_to_plot):]:
    ax.axis("off")
plt.tight_layout()
fig.savefig(figures_dir / "numerical_feature_distributions.png", dpi=150)
plt.show()

## 15. Correlation analysis (numerical features)

In [ ]:
corr = train_df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, cbar_kws={"shrink": 0.7})
ax.set_title("Correlation Matrix — Numerical Features")
plt.tight_layout()
fig.savefig(figures_dir / "correlation_matrix.png", dpi=150)
plt.show()